# From evolutionary data to verified prototypes — a walkthrough

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omega2417/bnt/blob/claude/software-project-publication-lz9nuu/evoproto/notebooks/evoproto_walkthrough.ipynb)

This notebook walks through the reference implementation (`evoproto`) of

> **From Evolutionary Biology Data to Technological Prototypes: An AI-Driven Engineering Design Framework with an Evidence-Gated Validation Protocol**
> Olga Torstensson, Yuriy Danyk, Dmytro Prokopovych-Tkachenko

It follows the paper section by section, runs every equation on live data and
plots what each one does. Nothing here is pre-baked: every number and every
figure below is computed when you run the cell.

**What the framework does.** Biologically inspired design usually copies a
single natural form. This framework instead treats the *record of how forms
changed under selection* — phylogenies, character matrices, fossil occurrences,
DOI-backed mechanism measurements — as structured constraints on AI-driven
generation of designs, and refuses to recommend when that record is too thin.

**The one thing to take away.** Every recommendation exposes a chain

```
Organism --hasTrait--> Trait --performs--> Function --explainedBy--> Mechanism --mapsTo--> EngParameter
```

with a provenance record on each of the four edges. Two numbers gate it: the
evidence score `E` (how well supported the chain is) and the transfer
uncertainty `U` (how far the biological setting is from the engineering one).
If `E` is too low or `U` too high, the system **abstains** and says which
experiment would fix that.

---

### How to use this notebook

Run the cells in order (`Runtime -> Run all` in Colab, about two minutes). Each
section states the equation it implements, runs it, and plots the result.

| section | topic | paper |
|---|---|---|
| 1 | setup | — |
| 2 | provenance, seeds, curation | §3, Eqs. (1)–(2) |
| 3 | the evolutionary–engineering knowledge graph | §4.2, Eqs. (3)–(4) |
| 4 | phylogenetic signal and convergence | §4.3, Eqs. (5)–(7) |
| 5 | analog retrieval and transfer uncertainty | §4.4, Eqs. (8)–(10) |
| 6 | the engineering case study | §6, Eqs. (15)–(19) |
| 7 | constrained multi-objective search | §4.5, Algorithm 1 |
| 8 | the evidence gate and abstention | §4.6, Eqs. (13)–(14) |
| 9 | the three-arm protocol and its statistics | §7 |
| 10 | ablations: what each source of value contributes | §7.5 |
| 11 | reproducibility and export | §5, §7.8 |

> **No biological records are bundled.** Everything executed here is tagged
> `SYNTHETIC`: the demonstration graph is hand-built and the trees and traits
> are simulated. The point is to make the machinery inspectable, not to assert
> anything about birds, bamboos or sea urchins.


## 1. Setup

The cell below imports `evoproto`, installing it from the repository if it is
not already present (that is what happens on a fresh Colab runtime). If you
opened this notebook from inside a clone, the local source is used instead, so
your edits take effect.


In [ ]:
import pathlib
import subprocess
import sys


def ensure_evoproto():
    # 1. already importable (installed, or a re-run of this cell)
    try:
        import evoproto
        return evoproto
    except ImportError:
        pass

    # 2. running from inside a clone: use the local sources
    here = pathlib.Path.cwd()
    for candidate in [here, *here.parents]:
        source = candidate / "src" / "evoproto" / "__init__.py"
        if source.exists():
            sys.path.insert(0, str(candidate / "src"))
            import evoproto
            print(f"using the local sources at {candidate / 'src'}")
            return evoproto

    # 3. fresh Colab runtime: install from the repository
    print("installing evoproto from the repository ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/omega2417/bnt.git#subdirectory=evoproto"],
        check=True,
    )
    import evoproto
    return evoproto


evoproto = ensure_evoproto()
print("evoproto", evoproto.__version__)


In [ ]:
import json
import textwrap

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 9,
    "legend.frameon": False,
})

# a small, colour-blind-safe palette used throughout the notebook
BLUE, RED, GREY, GREEN, SAND = "#2166ac", "#b2182b", "#7f7f7f", "#1b7837", "#dfc27d"
ARM_COLOUR = {"A": GREY, "B": BLUE, "C": RED}


def show(title=None):
    if title:
        plt.title(title)
    plt.tight_layout()
    plt.show()


## 2. Data foundation — provenance, seeds and curation (§3)

Three ideas do most of the work in this section.

**Every datum carries a provenance record.** `π = (source, version, licence,
tag, uri, date, r, u)`, where `r` flags a *reconstructed* ancestral state and
`u` is an uncertainty estimate. The tag is one of five values, and a quantity
derived from several records inherits the **weakest** of their tags — so a
modelled number can never be tabulated as a measured one by accident.

**Seeds are content-derived.** Python's built-in `hash()` is randomised per
process, so an experiment seeded from it is not reproducible across machines.
`stable_seed` hashes `(package version, task, arm, replicate)` with blake2b.

**Records are admitted, not assumed.** Eq. (1) measures how completely a taxon
is scored for the characters a query actually uses; Eq. (2) discounts a clade's
contribution by how thinly it is sampled.


In [ ]:
from evoproto.data import (
    CorpusSnapshot, Provenance, ProvenanceTag, admit_by_completeness,
    character_completeness, default_connectors, sampling_weight, stable_seed,
    taxon_completeness,
)

measured = Provenance("Gibson & Ashby (1997)", "2nd ed.", "CC-BY",
                      ProvenanceTag.MEASURED, uri="https://doi.org/10.1017/CBO9781139878326", u=0.15)
reconstructed = Provenance("ancestral state reconstruction", "demo", "CC0",
                           ProvenanceTag.MODELED, u=0.40, r=1, method="ML under Brownian motion")

print("a measured record :", measured.tag.value, "| r =", measured.r, "| u =", measured.u)
print("a reconstruction  :", reconstructed.tag.value, "| r =", reconstructed.r,
      "| method =", reconstructed.method)
print("tag inherited by a quantity derived from both:",
      Provenance.combine([measured, reconstructed]).value)

# a reconstruction must name its method - the framework refuses silent modelling
try:
    Provenance("somewhere", "1", "CC0", ProvenanceTag.MODELED, r=1)
except ValueError as error:
    print("refused:", error)


In [ ]:
print("seeds are derived from content, not from a randomised hash:")
for arm in ("A", "B", "C"):
    print(f"  task T1, arm {arm}, replicate 0 -> {stable_seed('T1', arm, 0)}")
print("  same call again          ->", stable_seed("T1", "C", 0), "(identical, on any machine)")


### Eq. (1) — missing-data control

A character matrix is taxa × characters with gaps. Completeness of taxon *i* is
the fraction of characters scored for it; a record enters retrieval only if it
clears `c_min` **for the characters the query uses**. That last qualifier
matters: a taxon scored sparsely overall can still be perfectly usable for a
query about the traits it *is* scored for.


In [ ]:
rng = np.random.default_rng(7)
n_taxa, n_characters = 18, 24
matrix = rng.normal(size=(n_taxa, n_characters))

# realistic missingness: some taxa are poorly scored, some characters rarely scored
taxon_quality = rng.beta(5, 2, size=n_taxa)
character_quality = rng.beta(6, 2, size=n_characters)
observed = rng.random((n_taxa, n_characters)) < np.outer(taxon_quality, character_quality)
matrix[~observed] = np.nan

query_characters = [2, 3, 4, 5, 6, 7]
c_min = 0.80
admitted = admit_by_completeness(matrix, c_min=c_min, query_characters=query_characters)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4), gridspec_kw={"width_ratios": [2, 1, 1]})

ax = axes[0]
ax.imshow(np.isnan(matrix), cmap="Greys", aspect="auto", interpolation="nearest")
ax.axvspan(min(query_characters) - 0.5, max(query_characters) + 0.5,
           color=RED, alpha=0.18, lw=0)
ax.set_xlabel("character"); ax.set_ylabel("taxon")
ax.set_title("missingness (black = not scored)\nred band = characters used by the query")
ax.grid(False)

ax = axes[1]
completeness = taxon_completeness(matrix[:, query_characters])
colours = [GREEN if ok else RED for ok in admitted]
ax.barh(np.arange(n_taxa), completeness, color=colours)
ax.axvline(c_min, color="black", ls="--", lw=1)
ax.set_xlabel("$c_i$ over the query characters"); ax.set_ylabel("taxon")
ax.set_title(f"Eq. (1): admitted if $c_i \\geq$ {c_min}")

ax = axes[2]
ax.bar(np.arange(n_characters), character_completeness(matrix), color=BLUE)
ax.set_xlabel("character"); ax.set_ylabel("$c_j$")
ax.set_title("character completeness")

show()
print(f"{admitted.sum()} of {n_taxa} taxa are admitted for this query "
      f"({(~admitted).sum()} fall below c_min)")


### Eq. (2) — sampling-bias control

Both morphological and fossil sampling are uneven across clades. When counting
how many times a mechanism evolved independently, each origin is weighted by
`w = min(1, sampled / accepted richness)`, so a conclusion resting on a clade
where 45 of 1000 species have been scored does not carry the same weight as one
resting on a thoroughly sampled group.


In [ ]:
from evoproto.casestudy import CLADE_SAMPLING

fig, ax = plt.subplots(figsize=(6.2, 3.0))
clades = list(CLADE_SAMPLING)
weights = [sampling_weight(*CLADE_SAMPLING[c]) for c in clades]
bars = ax.barh(clades, weights, color=BLUE)
for clade, bar in zip(clades, bars):
    sampled, accepted = CLADE_SAMPLING[clade]
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height() / 2,
            f"  {sampled:.0f} of {accepted:.0f} species", va="center", fontsize=8, color=GREY)
ax.set_xlabel("sampling weight $w_k$ (Eq. 2)")
ax.set_xlim(0, max(weights) * 2.4)
show("how much confidence each clade's evidence can carry")


In [ ]:
# The three source connectors are offline by default: they build request URLs,
# they do not fetch. The package can therefore be audited without network access.
from evoproto.data import OfflineError

connectors = default_connectors()
url, payload = connectors["otol"].match_names_request(["Gallus gallus", "Bambusa vulgaris"])
print("Open Tree of Life name resolution ->", url)
print("payload:", payload)
try:
    connectors["otol"].fetch(url, payload)
except OfflineError as error:
    print("\nrefused:", textwrap.fill(str(error), 88))


## 3. The evolutionary–engineering knowledge graph (§4.2)

Six node types, nine typed edge types, a provenance record and a support weight
on every edge. The schema is enforced *at insertion time*: an edge whose
endpoints have the wrong types is rejected rather than stored and discovered
later.

Two decisions distinguish this graph from earlier biologically-inspired-design
representations:

1. **Homology and convergence are mutually exclusive typed relations.** Whether
   two similar traits are related by descent or arose independently is decided
   by the phylogeny, not by how similar they look — and the graph will not let
   you assert both.
2. **The number of independent origins is derived, not declared.** Eq. (3)
   partitions the traits that reach a mechanism into homology classes with a
   union–find; the number of classes *is* the number of times the mechanism
   evolved.


In [ ]:
from evoproto.casestudy import build_demo_kg
from evoproto.kg import EdgeType, EvoKG, NodeType, SchemaError

kg = build_demo_kg()
print(kg)
print("\nnodes by type:")
for node_type in NodeType:
    names = kg.nodes_of_type(node_type)
    print(f"  {node_type.value:<13} {len(names)}  {', '.join(names)}")


In [ ]:
NODE_COLOUR = {
    NodeType.ENVIRONMENT: "#d9f0d3", NodeType.ORGANISM: "#c6dbef",
    NodeType.TRAIT: "#fdd0a2", NodeType.FUNCTION: "#dadaeb",
    NodeType.MECHANISM: "#fcbba1", NodeType.ENG_PARAMETER: "#d9d9d9",
}
# one column per node type, laid out in the order the chain of Eq. (4) travels
COLUMN = {
    NodeType.ENVIRONMENT: 0.0, NodeType.ORGANISM: 1.0, NodeType.TRAIT: 2.0,
    NodeType.FUNCTION: 3.0, NodeType.MECHANISM: 4.0, NodeType.ENG_PARAMETER: 5.0,
}
EDGE_STYLE = {
    EdgeType.HAS_TRAIT: ("black", "-"), EdgeType.LIVES_IN: (GREY, ":"),
    EdgeType.PERFORMS: ("black", "-"), EdgeType.EXPLAINED_BY: ("black", "-"),
    EdgeType.MAPS_TO: ("black", "-"), EdgeType.HOMOLOGOUS_TO: (BLUE, "-"),
    EdgeType.CONVERGENT_WITH: (RED, "--"), EdgeType.TRADES_OFF_WITH: (GREEN, "-."),
    EdgeType.DERIVED_FROM: (SAND, "-"),
}


def draw_kg(graph, ax, title=""):
    columns = {}
    for node in sorted(graph.graph.nodes):
        columns.setdefault(COLUMN[graph.node_type(node)], []).append(node)
    position = {}
    for x, nodes in columns.items():
        for i, node in enumerate(nodes):
            position[node] = (x, -(i - (len(nodes) - 1) / 2) * 1.35)

    seen = set()
    for tail, head, key in graph.graph.edges(keys=True):
        if (head, tail, key) in seen:      # draw each symmetric pair once
            continue
        seen.add((tail, head, key))
        colour, style = EDGE_STYLE[key]
        # edges inside one column (trait-trait, function-function) need curvature
        curve = 0.55 if position[tail][0] == position[head][0] else 0.05
        ax.annotate("", xy=position[head], xytext=position[tail],
                    arrowprops=dict(arrowstyle="-|>", color=colour, lw=1.2, ls=style,
                                    shrinkA=18, shrinkB=18, alpha=0.9,
                                    connectionstyle=f"arc3,rad={curve}"))

    for node, (x, y) in position.items():
        ax.scatter([x], [y], s=560, color=NODE_COLOUR[graph.node_type(node)],
                   edgecolor="black", linewidth=0.6, zorder=3)
        ax.text(x, y - 0.32, "\n".join(textwrap.wrap(node.replace("_", " "), 15)),
                ha="center", va="top", fontsize=6.4, zorder=4)

    top = max(y for _, y in position.values())
    for node_type, x in COLUMN.items():
        if x in columns:
            ax.text(x, top + 0.62, node_type.value, ha="center", fontsize=8.5, weight="bold")
    ax.set_xlim(-0.6, 5.6)
    ax.set_ylim(min(y for _, y in position.values()) - 1.1, top + 1.05)
    ax.set_axis_off()
    if title:
        ax.text(2.5, top + 1.02, title, ha="center", fontsize=9.5, style="italic")


fig, ax = plt.subplots(figsize=(13, 5.6))
draw_kg(kg, ax, "the demonstration EE-KG (SYNTHETIC)")
handles = [plt.Line2D([], [], color=c, ls=s, label=e.value) for e, (c, s) in EDGE_STYLE.items()]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0),
          ncol=5, fontsize=7.5)
plt.tight_layout()
plt.show()


Read the red dashed edges: the avian strut, the bamboo culm and the echinoid
stereom are marked **convergent**, while the two avian traits are **homologous**
to each other. That distinction is exactly what Eq. (3) counts.


In [ ]:
traits = kg.traits_of_mechanism("sandwich_stiffening")
classes = kg.homology_classes(traits)
print("traits reaching the mechanism 'sandwich_stiffening':")
for trait in traits:
    print("  ", trait)
print("\nhomology classes (Eq. 3 counts these):")
for i, members in enumerate(classes, start=1):
    print(f"  class {i}: {', '.join(members)}")

n_conv = kg.independent_origins("sandwich_stiffening")
n_conv_weighted = kg.weighted_independent_origins("sandwich_stiffening", CLADE_SAMPLING)
print(f"\nN_conv = {n_conv} independent origins")
print(f"N_conv^w = {n_conv_weighted:.3f} after the sampling correction of Eq. (2)")


In [ ]:
# What the schema refuses. These are the two failure modes the paper calls out:
# a structurally wrong edge, and asserting homology and convergence at once.
for description, action in [
    ("an Organism cannot 'perform' a Function directly",
     lambda: kg.add_edge("Aves.sp1", "bending_stiffness_per_mass", EdgeType.PERFORMS, 0.9, measured)),
    ("two traits already marked convergent cannot also be homologous",
     lambda: kg.add_edge("strutted_long_bone", "diaphragmed_culm", EdgeType.HOMOLOGOUS_TO, 0.9, measured)),
]:
    try:
        action()
        print("accepted (unexpected):", description)
    except SchemaError as error:
        print(f"refused - {description}\n    {error}\n")


### Eq. (4) — the traceability chain

Every recommendation must expose at least one complete path from an organism to
an engineering parameter, with all four provenance records. This is what a
design-review board reads instead of a black-box geometry.


In [ ]:
chains = kg.evidence_chains("core_relative_density")
print(f"{len(chains)} chains reach the engineering parameter 'core_relative_density'\n")

best = kg.best_chain("core_relative_density")
labels = ["hasTrait", "performs", "explainedBy", "mapsTo"]
for i, (node, label) in enumerate(zip(best.nodes, labels + [None])):
    print(f"  {node}")
    if label:
        record = best.provenance[i]
        print(f"      --{label}: s = {best.supports[i]:.2f}  [{record.tag.value}]  u = {record.u}")
print(f"\nfraction of the chain supported by measurement (M of Eq. 8): {best.measured_fraction:.2f}")
print(f"weakest edge: {labels[best.weakest_edge()]} (s = {min(best.supports):.2f})")


## 4. Phylogenetic context (§4.3)

Why a designer should care about phylogenetic signal: **a trait shared across a
clade because everyone inherited it is weak evidence that selection favoured it
for your function.** A trait with low signal that nevertheless keeps reappearing
in distant lineages is the opposite — a candidate for convergence, and therefore
for a mechanism that works repeatedly under similar pressures.

Three quantities make this operational:

* **Blomberg's K** (Eq. 5) — observed signal relative to what Brownian motion
  predicts on this tree. `K ≈ 1` is Brownian; `K > 1` means more clustering by
  ancestry than drift explains.
* **Pagel's λ** (Eq. 6) — a multiplier on the off-diagonal covariances,
  estimated by profile likelihood. `λ → 0` means the tree carries no information
  about the trait.
* **Stayton's C₁** (Eq. 7) — the fraction of ancestral divergence that later
  evolution has closed between two focal tips, with a p-value from Brownian
  simulations on the same tree (Eq. A.2).


In [ ]:
from evoproto import phylo

tree = phylo.random_ultrametric_tree(28, seed=3)
C = phylo.vcv_matrix(tree)
brownian, _ = phylo.simulate_brownian(tree, sigma2=1.0, seed=11)

rng = np.random.default_rng(5)
white_noise = rng.normal(scale=brownian.std(), size=len(tree.tips))

# A single draw is noisy, so look at several: K scatters widely around 1 for a
# Brownian trait, while white noise collapses both statistics towards zero.
print(f"{'trait':<22}{'K':>8}{'lambda':>9}")
for seed in range(11, 16):
    draw, _ = phylo.simulate_brownian(tree, sigma2=1.0, seed=seed)
    print(f"{'Brownian, seed ' + str(seed):<22}{phylo.blombergs_k(draw, C):>8.3f}"
          f"{phylo.pagels_lambda(draw, C):>9.3f}")
print(f"{'white noise':<22}{phylo.blombergs_k(white_noise, C):>8.3f}"
      f"{phylo.pagels_lambda(white_noise, C):>9.3f}")


In [ ]:
# Appendix A.1 states E[K] = 1 under Brownian motion on a given tree.
# That is a check we can actually run, not just quote.
ks = [phylo.blombergs_k(phylo.simulate_brownian(tree, 1.0, seed=s)[0], C) for s in range(400)]

fig, ax = plt.subplots(figsize=(6.0, 3.0))
ax.hist(ks, bins=40, color=BLUE, alpha=0.75)
ax.axvline(1.0, color="black", ls="--", lw=1.2, label="expectation under BM (Eq. A.1)")
ax.axvline(np.mean(ks), color=RED, lw=1.2, label=f"mean of 400 simulations = {np.mean(ks):.3f}")
ax.set_xlabel("Blomberg's K"); ax.set_ylabel("count"); ax.legend()
show("the implementation reproduces the analytic expectation")


In [ ]:
def plot_tree(ax, tree, highlight=(), trait=None):
    order = {tip: i for i, tip in enumerate(tree.tips)}
    y = dict(order)
    for node in tree.postorder():
        children = tree.children[node]
        if children:
            y[node] = float(np.mean([y[c] for c in children]))
    for node in range(tree.n_nodes):
        parent = int(tree.parent[node])
        if parent < 0:
            continue
        colour = RED if node in highlight else "black"
        width = 1.8 if node in highlight else 0.7
        ax.plot([tree.depth(parent), tree.depth(node)], [y[node], y[node]], color=colour, lw=width)
        ax.plot([tree.depth(parent)] * 2, [y[node], y[parent]], color="black", lw=0.6)
    if trait is not None:
        depth = max(tree.depth(t) for t in tree.tips)
        for tip in tree.tips:
            colour = RED if tip in highlight else BLUE
            ax.scatter([depth + 0.04], [y[tip]], s=18 + 60 * abs(trait[order[tip]]) / np.max(np.abs(trait)),
                       color=colour, zorder=3)
    ax.set_yticks([]); ax.grid(False)
    return y


# Induce convergence: force the two most distantly related tips to the same value.
shared = phylo.vcv_matrix(tree)
pairs = [(shared[i, j], i, j) for i in range(len(tree.tips)) for j in range(i + 1, len(tree.tips))]
_, i, j = min(pairs)
tip_i, tip_j = tree.tips[i], tree.tips[j]

converged = brownian.copy()
converged[i] = converged[j] = float(np.max(brownian) + 0.8)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
plot_tree(axes[0], tree, highlight=(tip_i, tip_j), trait=converged)
axes[0].set_xlabel("branch length from the root")
axes[0].set_title("(a) tree and trait; the two red tips\nare distant relatives forced together")

grid, loglik = phylo.lambda_profile(converged, C)
lam = phylo.pagels_lambda(converged, C)
axes[1].plot(grid, loglik, color=BLUE)
axes[1].axvline(lam, color=RED, lw=1.2)
axes[1].set_xlabel("Pagel's $\\lambda$"); axes[1].set_ylabel("profile log-likelihood")
axes[1].set_title(f"(b) Eq. (6): $\\hat\\lambda$ = {lam:.2f}, "
                  f"K = {phylo.blombergs_k(converged, C):.2f}")

observed_c1, p_value, null = phylo.c1_pvalue(tree, converged, tip_i, tip_j, n_sim=399, seed=1)
axes[2].hist(null, bins=35, color=GREY, alpha=0.7, label="$C_1$ under Brownian motion")
axes[2].axvline(observed_c1, color=RED, lw=1.5, label=f"observed $C_1$ = {observed_c1:.2f}")
axes[2].set_xlabel("Stayton's $C_1$"); axes[2].set_ylabel("count"); axes[2].legend(fontsize=7)
axes[2].set_title(f"(c) Eqs. (7) and (A.2): p = {p_value:.3f}")
show()

print(f"a convergentWith edge is inserted only if p <= 0.05 and the tips are not homologous"
      f" -> here p = {p_value:.3f}")


In [ ]:
# The same test on an ordinary Brownian trait must NOT find convergence.
# A method that always says "convergent" would be useless.
_, p_plain, _ = phylo.c1_pvalue(tree, brownian, tip_i, tip_j, n_sim=399, seed=2)
print(f"induced convergence : p = {p_value:.3f}  -> edge inserted")
print(f"plain Brownian trait: p = {p_plain:.3f}  -> no edge")


### Ancestral states

`C₁` needs reconstructed ancestral phenotypes, which the package computes by
two-pass pruning under Brownian motion. Reconstructions are model outputs, so
records built from them carry `r = 1` and are discounted later by Eq. (13).
The plot below shows how far the reconstruction is from the truth in a
simulation where, unusually, we know the truth.


In [ ]:
tips_observed, truth = phylo.simulate_brownian(tree, sigma2=1.0, seed=21)
estimated = phylo.ancestral_states(tree, tips_observed)
internal = [n for n in range(tree.n_nodes) if n not in tree.tips]

fig, ax = plt.subplots(figsize=(4.6, 4.2))
ax.scatter(truth[internal], estimated[internal], s=22, color=BLUE, alpha=0.85)
limits = [min(truth[internal].min(), estimated[internal].min()) - 0.3,
          max(truth[internal].max(), estimated[internal].max()) + 0.3]
ax.plot(limits, limits, color="black", ls="--", lw=1)
ax.set_xlabel("true ancestral state (simulated)"); ax.set_ylabel("reconstructed state")
rmse = float(np.sqrt(np.mean((estimated[internal] - truth[internal]) ** 2)))
show(f"reconstruction error: RMSE = {rmse:.2f}\ndeeper nodes are the uncertain ones")


## 5. Analog retrieval and transfer uncertainty (§4.4)

An analog is scored by

$$S(c) = w_F F(c) + w_M M(c) + w_C \ln(1 + N^w_{\mathrm{conv}}(c)) - w_U U(c)$$

with `w_F = 0.40`, `w_M = 0.30`, `w_C = 0.15`, `w_U = 0.40` fixed before the
experiment. `F` is functional match, `M` the fraction of the chain backed by
measurement, `N^w_conv` the sampling-weighted count of independent origins from
Eq. (3). The logarithm encodes diminishing returns: the third independent origin
of a mechanism tells you less than the second did.

`U` (Eq. 9) is what stops a pretty analogy from becoming a bad part:

$$U = \alpha\,u_{\mathrm{scale}} + \beta\,u_{\mathrm{mat}} + \gamma\,u_{\mathrm{load}},
\qquad u_{\mathrm{scale}} = 1 - \exp(-|\log_{10}(\ell_{\mathrm{eng}}/\ell_{\mathrm{bio}})|)$$

One decade of size mismatch already costs 0.63. That is deliberate: surface-to-
volume ratios, buckling loads and viscous-versus-inertial regimes all change
with size, so shape copying across orders of magnitude is the classic way to get
a biomimetic part wrong.


In [ ]:
from evoproto.retrieval import (
    LOAD_REGIMES, MATERIAL_CLASSES, RetrievalWeights, TransferWeights,
    GroundedStatement, filter_grounded_statements, rank_analogs, score_analog,
    transfer_uncertainty, u_scale,
)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.4), gridspec_kw={"width_ratios": [1.1, 1.4]})

ax = axes[0]
decades = np.linspace(-3, 3, 400)
ax.plot(decades, [u_scale(10.0 ** d, 1.0) for d in decades], color=BLUE, lw=1.6)
for d, label in [(1, "one decade\n0.63"), (2, "two decades\n0.86")]:
    ax.scatter([d], [u_scale(10.0 ** d, 1.0)], color=RED, zorder=3)
    ax.annotate(label, (d, u_scale(10.0 ** d, 1.0)), textcoords="offset points",
                xytext=(6, -16), fontsize=7.5, color=RED)
ax.set_xlabel("$\\log_{10}(\\ell_{eng}/\\ell_{bio})$"); ax.set_ylabel("$u_{scale}$")
ax.set_title("Eq. (10): the cost of a scale jump")

ax = axes[1]
names = list(MATERIAL_CLASSES)
distance = np.abs(np.array(list(MATERIAL_CLASSES.values()))[:, None]
                  - np.array(list(MATERIAL_CLASSES.values()))[None, :])
image = ax.imshow(distance, cmap="YlOrRd", vmin=0, vmax=1)
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=40, ha="right", fontsize=7)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
ax.grid(False)
plt.colorbar(image, ax=ax, fraction=0.046).set_label("$u_{mat}$", fontsize=8)
ax.set_title("material behaviour classes (datasheet Table D1)")
show()

print("loading regimes (Table D2):", ", ".join(f"{k} = {v}" for k, v in LOAD_REGIMES.items()))


### The three candidate analogs of the case study (§6.3)

The functional query is *"maximise bending stiffness per unit mass of a slender
cantilevered member under intermittent inertial load"*. It retrieves three trait
families from mutually distant lineages. They are **candidates, not
conclusions**: each has to pass the phylogenetic tests above and have DOI-backed
mechanism support before it contributes to the score.


In [ ]:
from evoproto.casestudy import FUNCTIONAL_QUERY, candidate_analogs

print(textwrap.fill(FUNCTIONAL_QUERY, 88), "\n")
analogs = candidate_analogs(kg)
ranked = rank_analogs(analogs)

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.6))
labels = [r["identifier"] for r in ranked]
y = np.arange(len(ranked))
weights = RetrievalWeights()
transfer = TransferWeights()

ax = axes[0]
terms = {
    "$w_F F$": [weights.w_F * r["F"] for r in ranked],
    "$w_M M$": [weights.w_M * r["M"] for r in ranked],
    "$w_C \\ln(1+N^w)$": [weights.w_C * r["convergence_term"] for r in ranked],
}
left = np.zeros(len(ranked))
for (label, values), colour in zip(terms.items(), [BLUE, GREEN, SAND]):
    ax.barh(y, values, left=left, color=colour, label=label)
    left += np.asarray(values)
penalty = [-weights.w_U * r["U"] for r in ranked]
ax.barh(y, penalty, left=left, color=RED, label="$-w_U U$")
ax.scatter([r["score"] for r in ranked], y, color="black", zorder=4, s=28, label="$S(c)$")
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8); ax.invert_yaxis()
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("contribution to the score"); ax.legend(fontsize=7, loc="lower right")
ax.set_title("Eq. (8) decomposed")

ax = axes[1]
components = ["u_scale", "u_mat", "u_load"]
coefficients = [transfer.alpha, transfer.beta, transfer.gamma]
left = np.zeros(len(ranked))
for component, coefficient, colour in zip(components, coefficients, [BLUE, RED, GREEN]):
    values = [coefficient * r["U_components"][component] for r in ranked]
    ax.barh(y, values, left=left, color=colour, label=f"{component} x {coefficient}")
    left += np.asarray(values)
ax.axvline(0.5, color="black", ls="--", lw=1.2)
ax.text(0.5, -0.65, r"  $\tau_U$ = 0.5", fontsize=8)
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8); ax.invert_yaxis()
ax.set_xlabel("transfer uncertainty $U$"); ax.legend(fontsize=7)
ax.set_title("Eq. (9): where the uncertainty comes from")
show()


The bamboo culm wins, and the reason is legible in the right-hand panel: it is
the closest in *scale* to a 50 mm bracket section. The echinoid stereom has the
friendlier material class (a mineral foam is closer to a printed lattice than a
hierarchical composite is) but sits a decade away in size, and the avian bone
loses on both counts. `U = 0.38` for the winner, comfortably below `τ_U = 0.5`
— this is the number quoted in §6.3 of the paper.

### Grounding the language model

The retrieval-augmented stage is where a language model is asked to state *how*
the biological mechanism corresponds to the engineering requirement. That is
also where it is most likely to invent a plausible mechanism. The mitigation is
not a prompt instruction but a filter: a statement citing anything outside the
retrieved subgraph is discarded, and so is one that cites nothing at all.


In [ ]:
allowed = {"10.1017/CBO9781139878326", "edge:explainedBy:sandwich_stiffening"}
statements = [
    GroundedStatement("bamboo_culm",
                      "The culm wall acts as a stiff shell around a low-density core, so "
                      "bending stiffness scales faster than mass.",
                      ("10.1017/CBO9781139878326",)),
    GroundedStatement("bamboo_culm",
                      "Bamboo nodes contain a piezoelectric layer that stiffens under load.",
                      ("10.9999/not-in-the-corpus",)),
    GroundedStatement("avian_long_bone",
                      "Avian bones are known to be light.", ()),
]
kept, discarded = filter_grounded_statements(statements, allowed)
for statement in kept:
    print("KEPT     ", textwrap.shorten(statement.text, 76), "|", statement.citations)
for statement in discarded:
    reason = "cites outside the retrieved set" if statement.citations else "no citation"
    print("DISCARDED", textwrap.shorten(statement.text, 76), f"| {reason}")


## 6. The engineering case study (§6)

An end-of-arm-tool bracket connecting a vacuum gripper to a four-axis
pick-and-place robot. Cycle time on such a line is limited by the acceleration
the robot can apply to the mass at the wrist, so every gram removed is either a
faster cycle or a lower motor current.

The bracket is idealised as a cantilever under a tip load `P = m_p a`, with a
rectangular shell section of width `b`, height `h` and wall `t`, filled with a
cellular core of relative density `ρ`. The evaluator implements Eqs. (15)–(19);
the core follows the Gibson–Ashby open-cell scaling `E* = C₁ E_s ρ²`.

**All values are assumptions A1–A7 of Table 6, not measurements.** They are
placeholders for the industrial partner's certified data.


In [ ]:
from evoproto import design

spec = design.EOAT_BRACKET
print(f"cantilever length L     : {spec.L * 1000:.0f} mm")
print(f"payload x acceleration  : {spec.payload:.1f} kg x {spec.acceleration:.1f} m/s^2"
      f"  ->  P = {spec.tip_load:.1f} N")
print(f"allowable deflection    : {spec.delta_max * 1e6:.0f} um")
print(f"allowable stress        : {spec.sigma_allow / 1e6:.0f} MPa (yield / SF)")
print(f"mass cap                : {spec.mass_max * 1000:.0f} g")
print(f"minimum printable wall  : {spec.t_min * 1000:.1f} mm")

example = design.evaluate([0.012, 0.070, 0.0006, 0.08], spec)
print("\nan example design b=12 mm, h=70 mm, t=0.6 mm, rho=0.08:")
print(f"  mass       {example['mass'] * 1000:8.1f} g")
print(f"  deflection {example['deflection'] * 1e6:8.1f} um")
print(f"  stress     {example['stress'] / 1e6:8.1f} MPa")
print(f"  feasible   {example['feasible']}   [{example['tag']}]")
print("  margins:", {k: round(v, 4) for k, v in example["constraints"].items()})


In [ ]:
# How hard is this task? Sample the design box uniformly and look at what survives.
lo, hi = design.design_bounds(spec)
rng = np.random.default_rng(0)
X = lo + rng.random((20000, 4)) * (hi - lo)
out = design.evaluate_population(X, spec)
feasible = out["feasible"]

fig, axes = plt.subplots(1, 2, figsize=(11.5, 3.9))

ax = axes[0]
ax.scatter(out["mass"][~feasible] * 1000, out["deflection"][~feasible] * 1e6,
           s=3, color=GREY, alpha=0.25, label="infeasible")
ax.scatter(out["mass"][feasible] * 1000, out["deflection"][feasible] * 1e6,
           s=7, color=GREEN, label="feasible")
ax.axvline(spec.mass_max * 1000, color=RED, ls="--", lw=1.1)
ax.axhline(spec.delta_max * 1e6, color=RED, ls="--", lw=1.1)
ax.set_xlim(0, 400); ax.set_ylim(0, 600)
ax.set_xlabel("mass [g]"); ax.set_ylabel("tip deflection [$\\mu$m]")
ax.legend(fontsize=7.5)
ax.set_title(f"20 000 uniform samples: {100 * feasible.mean():.2f} % feasible\n"
             "(the paper reports about 1 %)")

ax = axes[1]
h_grid = np.linspace(lo[1], hi[1], 90)
rho_grid = np.linspace(lo[3], hi[3], 90)
H, R = np.meshgrid(h_grid, rho_grid)
probe = np.column_stack([np.full(H.size, 0.020), H.ravel(),
                         np.full(H.size, 0.0008), R.ravel()])
probe_out = design.evaluate_population(probe, spec)
field = probe_out["feasible"].reshape(H.shape)
ax.contourf(H * 1000, R, field, levels=[-0.5, 0.5, 1.5], colors=["#f0f0f0", "#c7e9c0"])
mass_field = probe_out["mass"].reshape(H.shape) * 1000
contours = ax.contour(H * 1000, R, mass_field, levels=[40, 60, 80, 100, 120],
                      colors="black", linewidths=0.7)
ax.clabel(contours, fmt="%.0f g", fontsize=7)
ax.set_xlabel("section height $h$ [mm]"); ax.set_ylabel("core relative density $\\rho$")
ax.set_title("slice at $b$ = 20 mm, $t$ = 0.8 mm\ngreen = feasible, contours = mass")
show()


The right-hand panel is the mechanism made visible: the feasible region is a
narrow band of **tall sections with a light core** — exactly the sandwich
stiffening the biological analogs point at. This is why the evolutionary data
can help at all: it shapes where the search starts looking.

## 7. Constrained multi-objective search (§4.5, Algorithm 1)

Three arms, identical evaluator, identical budget, identical seeds:

* **Arm A** — a template-ratio stand-in for expert biomimetic practice:
  remembered proportions (height a catalogue multiple of the width, a handbook
  wall fraction, a solid-ish core).
* **Arm B** — generative design with a uniform prior, no knowledge-graph access.
* **Arm C** — the proposed framework: the analog-shaped prior `p_c`.

Arms B and C differ **only** in the prior, so any difference between them is
attributable to the evolutionary data and to nothing else.


In [ ]:
from evoproto import optimize
from evoproto.data import stable_seed


def evaluator(X):
    out = design.evaluate_population(X, spec)
    return out["F"], out["G"]


def run_arm(arm, replicate=0, population=40, generations=25):
    prior = design.PRIORS[arm]
    return optimize.search(
        evaluator,
        lambda n, rng: prior(n, spec, rng),
        lo, hi,
        population_size=population, generations=generations,
        seed=stable_seed("T1", arm, replicate), arm=arm,
        projection=design.template_projection if arm == "A" else None,
    )


runs = {arm: [run_arm(arm, replicate) for replicate in range(10)] for arm in ("A", "B", "C")}
print("done:", {arm: len(results) for arm, results in runs.items()},
      "| evaluations per run:", runs["C"][0].evaluations)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.7))

ax = axes[0]
for arm, results in runs.items():
    F = results[0].objectives
    ax.scatter(F[:, 0] * 1000, F[:, 1] * 1e6, s=14, alpha=0.8,
               color=ARM_COLOUR[arm], label=f"arm {arm}")
ax.axvline(spec.mass_max * 1000, color="black", ls="--", lw=0.9)
ax.axhline(spec.delta_max * 1e6, color="black", ls="--", lw=0.9)
ax.set_xlim(0, 260); ax.set_ylim(0, 320)
ax.set_xlabel("mass [g]"); ax.set_ylabel("tip deflection [$\\mu$m]")
ax.legend(fontsize=7.5)
ax.set_title("(a) final population, replicate 0\ndashed = specification limits")

ax = axes[1]
for arm, results in runs.items():
    curves = np.array([r.feasible_fraction_per_generation for r in results])
    generations = np.arange(1, curves.shape[1] + 1)
    ax.plot(generations, curves.mean(axis=0), color=ARM_COLOUR[arm], lw=1.6, label=f"arm {arm}")
    ax.fill_between(generations, curves.min(axis=0), curves.max(axis=0),
                    color=ARM_COLOUR[arm], alpha=0.15, lw=0)
ax.set_xlabel("generation"); ax.set_ylabel("feasible fraction of the population")
ax.legend(fontsize=7.5)
ax.set_title("(b) how fast each arm reaches\na satisfiable design")

ax = axes[2]
data, positions = [], []
for position, (arm, results) in enumerate(runs.items()):
    values = [optimize.normalized_hypervolume(r.archive_objectives, spec.reference_point)
              if r.archive.shape[0] else 0.0 for r in results]
    data.append(values); positions.append(position + 1)
boxes = ax.boxplot(data, widths=0.55, patch_artist=True)
for patch, arm in zip(boxes["boxes"], runs):
    patch.set_facecolor(ARM_COLOUR[arm]); patch.set_alpha(0.5)
for element in ("medians", "whiskers", "caps"):
    for item in boxes[element]:
        item.set_color("black")
ax.set_xticks(positions); ax.set_xticklabels(list(runs))
ax.set_xlabel("arm"); ax.set_ylabel("normalised hypervolume")
ax.set_title("(c) front quality over 10 replicates")
show()


Arm A's designs cluster beyond the mass cap: a remembered proportion with a
solid core simply cannot meet a 120 g budget at this stiffness. Arms B and C
both find the feasible band; C gets there faster (panel b) and ends marginally
ahead on hypervolume (panel c).

**This is a synthetic comparison and proves nothing about the hypothesis.** Arm
A is a stand-in, not a human designer, and arm C's prior is hand-set, not
derived from a curated corpus. What it does show is that the measurement
apparatus works and can register differences between arms.


## 8. The evidence gate (§4.6) — the part that can say no

A design that passes numerical verification is still not a recommendation. The
framework also asks whether the biological justification is strong enough to be
worth putting in front of an engineer:

$$E = \min\left\{1,\; \Big(\sum_{k=1}^{4} w_k s_k\Big)\big(1 + \kappa \ln(1 + N^w_{\mathrm{conv}})\big)\,\rho_{\mathrm{rec}}\right\}$$

with `κ = 0.15` and `ρ_rec = 0.7` when the chain rests only on reconstructed
states. Then

```
REJECT     if verification failed
ABSTAIN    if E < τ_E  or  U > τ_U
RECOMMEND  otherwise — subject to engineer approval
```

An abstention returns a **missing-evidence report**: which edge is weakest,
which component of `U` blew the budget, and the experiment that would fix it.
That converts a gap in evidence into a research task instead of a dead end.


In [ ]:
from evoproto.gate import Decision, EvidenceWeights, GateThresholds, decide, evidence_score
from evoproto.kg import EvidenceChain

thresholds = GateThresholds()
print(f"thresholds: tau_E = {thresholds.tau_E}, tau_U = {thresholds.tau_U}")
print(f"E of the case-study chain {tuple(round(s, 2) for s in best.supports)} "
      f"with N_conv^w = {n_conv_weighted:.3f}: {evidence_score(best.supports, n_conv_weighted):.3f}")
print(f"the same chain if every record were a reconstruction: "
      f"{evidence_score(best.supports, n_conv_weighted, reconstruction_only=True):.3f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))

# (a) the decision map, with the case study on it
ax = axes[0]
ax.add_patch(plt.Rectangle((thresholds.tau_E, 0), 1 - thresholds.tau_E, thresholds.tau_U,
                           facecolor="#d6ecd2", lw=0))
ax.add_patch(plt.Rectangle((0, 0), thresholds.tau_E, 1, facecolor="#faeccc", lw=0))
ax.add_patch(plt.Rectangle((thresholds.tau_E, thresholds.tau_U),
                           1 - thresholds.tau_E, 1 - thresholds.tau_U, facecolor="#faeccc", lw=0))
ax.axvline(thresholds.tau_E, color="black", ls="--", lw=1)
ax.axhline(thresholds.tau_U, color="black", ls="--", lw=1)
ax.text(0.75, 0.12, "RECOMMEND\n(awaiting engineer approval)", ha="center", fontsize=8)
ax.text(0.25, 0.72, "ABSTAIN\nevidence too weak", ha="center", fontsize=8)
ax.text(0.75, 0.75, "ABSTAIN\ntransfer too uncertain", ha="center", fontsize=8)

E_case = evidence_score(best.supports, n_conv_weighted)
U_case = ranked[0]["U"]
ax.scatter([E_case], [U_case], s=70, color=RED, zorder=5, edgecolor="white")
ax.annotate("case study\n(bamboo culm analog)", (E_case, U_case),
            textcoords="offset points", xytext=(-62, -30), fontsize=7.5, color=RED, ha="center",
            arrowprops=dict(arrowstyle="-", color=RED, lw=0.8))
ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.grid(False)
ax.set_xlabel("evidence score $E$ (Eq. 13)"); ax.set_ylabel("transfer uncertainty $U$ (Eq. 9)")
ax.set_title("(a) Eq. (14): the decision map")

# (b) what happens to E as the mechanism edge loses its support
ax = axes[1]
mechanism_weighted = EvidenceWeights(w=(0.15, 0.15, 0.50, 0.20))
mechanism_support = np.linspace(0.0, 1.0, 120)
for weights, label, style in [
    (EvidenceWeights(), "uniform edge weights (shipped default)", "-"),
    (mechanism_weighted, "mechanism-weighted gate $w_3$ = 0.50", "--"),
]:
    scores = [
        evidence_score((best.supports[0], best.supports[1], s, best.supports[3]),
                       n_conv_weighted, weights=weights)
        for s in mechanism_support
    ]
    ax.plot(mechanism_support, scores, ls=style, lw=1.6, label=label)
ax.axhline(thresholds.tau_E, color=RED, ls="--", lw=1.2)
ax.text(0.02, thresholds.tau_E + 0.015, r"$\tau_E$", color=RED, fontsize=9)
ax.axvline(best.supports[2], color=GREY, lw=1, ls=":")
ax.text(best.supports[2] - 0.03, 0.93, "current mechanism\nsupport", fontsize=7,
        color=GREY, ha="right", va="top")
ax.set_xlabel("support of the explainedBy edge ($s_3$)"); ax.set_ylabel("evidence score $E$")
ax.set_ylim(0, 1.0); ax.legend(fontsize=7.5, loc="lower right")
ax.set_title("(b) degrading the mechanism edge:\nwhen can the gate still abstain?")
show()


Panel (b) is the practically useful one, and it exposes a property of the
specification worth flagging. Because `E` is a **weighted mean** over the four
edges, with the shipped uniform weights the solid curve never reaches `τ_E`:
even with *zero* measured mechanism support, three strong edges keep the score
at 0.62 and the framework still recommends. Only if the gate is configured to
emphasise the mechanism edge (dashed, `w₃ = 0.50`) does an unmeasured mechanism
actually trigger abstention.

That is not a bug in the implementation — Eq. (13) is a weighted sum and is
implemented as one — but it does mean the shipped uniform weights make the gate
hard to trip on a single missing measurement. A deployment that wants
"no measured mechanism ⇒ abstain" must either weight `w₃` heavily or add a
minimum-support rule alongside the mean. This is the kind of thing worth fixing
in the pre-registration, before results exist to argue about.

### The three decisions, side by side


In [ ]:
from evoproto.data import Provenance, ProvenanceTag

thin = Provenance("thin corpus", "demo", "CC0", ProvenanceTag.EXTERNAL, u=0.6)


def show_decision(title, chain, U, components, verified=True, n_conv=n_conv_weighted):
    result = decide(chain=chain, U=U, n_conv_weighted=n_conv, verified=verified,
                    uncertainty_components=components, thresholds=thresholds)
    print(f"--- {title}")
    print(f"    E = {result.E if result.E == result.E else float('nan'):.3f}   "
          f"U = {result.U:.3f}   ->  {result.decision.value}")
    if result.report:
        print("   ", textwrap.fill(result.report.reason, 84, subsequent_indent="    "))
        print("    next:", textwrap.fill(result.report.suggested_experiment, 80,
                                         subsequent_indent="          "))
    print()


components_case = ranked[0]["U_components"]

show_decision("well-supported chain, transfer within budget", best, U_case, components_case)

weak_chain = EvidenceChain(nodes=best.nodes, supports=(0.62, 0.48, 0.08, 0.40),
                           provenance=(thin,) * 4)
show_decision("the mechanism has never been measured", weak_chain, U_case, components_case)

show_decision("a good chain, but transferred across three decades of scale", best, 0.71,
              {"u_scale": 0.86, "u_mat": 0.80, "u_load": 0.35})

show_decision("numerical verification failed", best, U_case, components_case, verified=False)


Note the asymmetry the framework is built around: the *strongest* output is a
recommendation with a traceability chain that still needs a human signature; the
*weakest* is not silence but a named experiment.

## 9. The three-arm protocol and its statistics (§7)

The pre-registered plan: three arms × three task families × ten seeded
replicates, four metrics (feasible-solution rate, normalised hypervolume, design
diversity, iterations to specification), paired two-sided Wilcoxon signed-rank
tests, Holm correction at family-wise 0.05, Cliff's δ and bootstrap confidence
intervals.

The tests are **two-sided even though the predictions are directional**, so that
evidence against the framework is reported with the same weight as evidence for
it. Task T1 (the bracket) is the only task with a fully specified evaluator
here; T2 and T3 await the partner's solvers and raise `NotImplementedError`
rather than quietly returning something.


In [ ]:
from evoproto import experiment

report = experiment.dry_run(replicates=10)
print(report["warning"], "\n")

header = f"{'arm':<5}{'FSR':>18}{'HV':>18}{'D':>18}{'ITS':>18}"
print(header); print("-" * len(header))
for cell, metrics in sorted(report["summary"].items()):
    row = f"{cell.split('/')[1]:<5}"
    for metric in ("FSR", "HV", "D", "ITS"):
        row += f"{metrics[metric]['mean']:>11.3f} +- {metrics[metric]['sd']:<4.3f}"
    print(row)


In [ ]:
cells = report["cells"]
metrics = ["FSR", "HV", "D", "ITS"]
fig, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, metric in zip(axes, metrics):
    data = [[c[metric] for c in cells if c["arm"] == arm] for arm in ("A", "B", "C")]
    boxes = ax.boxplot(data, widths=0.6, patch_artist=True)
    for patch, arm in zip(boxes["boxes"], ("A", "B", "C")):
        patch.set_facecolor(ARM_COLOUR[arm]); patch.set_alpha(0.5)
    for element in ("medians", "whiskers", "caps"):
        for item in boxes[element]:
            item.set_color("black")
    for position, values in enumerate(data, start=1):
        jitter = np.random.default_rng(position).normal(0, 0.04, len(values))
        ax.scatter(position + jitter, values, s=9, color="black", alpha=0.5, zorder=3)
    ax.set_xticks([1, 2, 3]); ax.set_xticklabels(["A", "B", "C"])
    direction = "higher is better" if metric != "ITS" else "lower is better"
    ax.set_title(f"{metric}\n({direction})", fontsize=9)
show()


In [ ]:
# Paired differences with bootstrap confidence intervals: the effect-size view
# that the plan requires alongside every p-value.
rows = report["comparisons"]
fig, ax = plt.subplots(figsize=(8.2, 3.6))
labels, centres, lows, highs, colours = [], [], [], [], []
for row in rows:
    # normalise each metric by its own scale so the contrasts are comparable
    scale = max(abs(row["ci_low"]), abs(row["ci_high"]), abs(row["median_difference"]), 1e-9)
    labels.append(f"{row['contrast']}  {row['metric']}")
    centres.append(row["median_difference"] / scale)
    lows.append(row["ci_low"] / scale)
    highs.append(row["ci_high"] / scale)
    colours.append(GREEN if row["p_holm"] <= 0.05 else GREY)

y = np.arange(len(labels))
for position, centre, low, high, colour in zip(y, centres, lows, highs, colours):
    ax.plot([low, high], [position, position], color=colour, lw=2.2, solid_capstyle="butt")
    ax.plot([low, low], [position - 0.16, position + 0.16], color=colour, lw=1.4)
    ax.plot([high, high], [position - 0.16, position + 0.16], color=colour, lw=1.4)
ax.scatter(centres, y, color=colours, zorder=3, s=34)
ax.axvline(0, color="black", lw=1)
ax.set_yticks(y); ax.set_yticklabels(labels, fontsize=8); ax.invert_yaxis()
ax.set_xlabel("median paired difference, scaled per metric (95 % bootstrap CI)")
ax.set_title("green = significant after Holm correction")
show()

print(f"{'contrast':<10}{'metric':<6}{'p_raw':>9}{'p_holm':>9}{'delta':>8}  effect")
for row in rows:
    print(f"{row['contrast']:<10}{row['metric']:<6}{row['p_raw']:>9.4f}{row['p_holm']:>9.4f}"
          f"{row['cliffs_delta']:>+8.2f}  {row['effect_magnitude']}")


Read the C-versus-B rows carefully. Hypervolume is significant after correction
with a large effect; **design diversity is not significant and points the wrong
way** (negative δ — arm C is *less* diverse than arm B here). That is exactly
the kind of outcome the real experiment must be able to report, and it is why
the dry run is worth publishing: a plan that could only produce favourable
results would not be a test.

### How much can ten replicates actually detect?

Eq. (24) gives the number of pairs needed for a standardised effect `d` at
α = 0.05 two-sided and 80 % power, adjusted for the Wilcoxon test's asymptotic
relative efficiency (the `π/3` factor).


In [ ]:
effects = np.linspace(0.4, 1.6, 200)
needed = [experiment.required_replicates(d) for d in effects]

fig, ax = plt.subplots(figsize=(6.2, 3.2))
ax.plot(effects, needed, color=BLUE, lw=1.6)
ax.axhline(10, color=RED, ls="--", lw=1.2)
ax.text(0.42, 11, "the protocol's 10 replicates", color=RED, fontsize=8)
for d in (0.8, 1.0):
    n = experiment.required_replicates(d)
    ax.scatter([d], [n], color=RED, zorder=3)
    ax.annotate(f"d = {d}: n = {n}", (d, n), textcoords="offset points",
                xytext=(8, 6), fontsize=8)
ax.set_xlabel("standardised effect size $d$"); ax.set_ylabel("required pairs $n$")
ax.set_ylim(0, 60)
show("Eq. (24): ten replicates detect large effects only")


This is a limitation the paper states rather than hides (§8.3): with ten
replicates a medium effect would very likely be missed, and the protocol allows
extension to twenty if the interim variance in arm B — which is independent of
the hypothesis — turns out larger than assumed.

## 10. Ablations — what each source of value actually contributes (§7.5)

The framework claims several distinct mechanisms of benefit. Each can be
switched off independently, which is the only honest way to find out whether it
does anything. Some act on the *decision* (through the terms of Eq. 13) and some
on the *search* (through the prior), so they show up in different places.

One caution about doing this correctly: the baseline uses the sampling-weighted
origin count, so "remove the convergence evidence" has to mean *one clade's
weight*, not the raw number 1 — otherwise the ablation silently switches scales
and appears to improve the score it was meant to degrade.


In [ ]:
from evoproto.data import Provenance, ProvenanceTag as Tag

# The chain used below runs through Aves, so "only one origin counts" means the
# sampling weight of that one clade - not the raw count 1, which would silently
# mix the weighted and unweighted scales and make the ablation look beneficial.
single_origin = sampling_weight(*CLADE_SAMPLING["Aves"])
reconstruction = Provenance("ancestral reconstruction", "demo", "CC0", Tag.MODELED,
                            u=0.4, r=1, method="ML under BM")
reconstruction_chain = EvidenceChain(nodes=best.nodes, supports=best.supports,
                                     provenance=(reconstruction,) * 4)
unmeasured_chain = EvidenceChain(nodes=best.nodes,
                                 supports=(best.supports[0], best.supports[1], 0.10, best.supports[3]),
                                 provenance=best.provenance)

uniform = EvidenceWeights()
scenarios = [
    ("baseline", best, n_conv_weighted, uniform),
    ("(a) no convergence evidence", best, single_origin, uniform),
    ("(b) chain rests only on reconstructions", reconstruction_chain, n_conv_weighted, uniform),
    ("(a)+(b) together", reconstruction_chain, single_origin, uniform),
    ("(c) mechanism never measured", unmeasured_chain, n_conv_weighted, uniform),
    ("(c) with a mechanism-weighted gate", unmeasured_chain, n_conv_weighted, mechanism_weighted),
]

results = []
for label, chain, n, weights in scenarios:
    E = evidence_score(chain.supports, n, chain.reconstruction_only, weights=weights)
    verdict = decide(chain, U_case, n, True, components_case, thresholds, weights=weights).decision
    results.append((label, E, verdict))
    print(f"{label:<42}E = {E:.3f}   N^w = {n:.3f}   -> {verdict.value}")

without_reconstructions = kg.without_reconstructions()
print(f"\nremoving every r = 1 edge from the graph drops "
      f"{kg.graph.number_of_edges() - without_reconstructions.graph.number_of_edges()} of "
      f"{kg.graph.number_of_edges()} edges")


In [ ]:
fig, ax = plt.subplots(figsize=(8.4, 3.4))
labels = [r[0] for r in results]
values = [r[1] for r in results]
colours = [GREEN if r[2] is Decision.RECOMMEND else SAND for r in results]
bars = ax.barh(np.arange(len(results)), values, color=colours)
ax.axvline(thresholds.tau_E, color=RED, ls="--", lw=1.3)
ax.text(thresholds.tau_E + 0.008, len(results) - 0.4, r"$\tau_E$", color=RED, fontsize=9)
for bar, (_, E, verdict) in zip(bars, results):
    ax.text(E + 0.012, bar.get_y() + bar.get_height() / 2, f"{E:.3f}  {verdict.value}",
            va="center", fontsize=7.5)
ax.set_yticks(np.arange(len(results))); ax.set_yticklabels(labels, fontsize=8)
ax.invert_yaxis(); ax.set_xlim(0, 1.15); ax.set_xlabel("evidence score $E$")
show("green = still recommended, sand = abstained")


Three things this ablation actually shows, including one that is awkward for the
framework:

* Removing the convergence evidence barely moves `E` (0.837 → 0.834). The reason
  is worth stating plainly: Eq. (2) weights each origin by the *fraction* of a
  clade that has been sampled, so with realistic figures — 180 of 11 000 avian
  species — the weighted count is `N^w_conv ≈ 0.10` rather than 3, and the
  convergence bonus `κ ln(1 + N^w_conv)` contributes about **1 %** of the
  evidence score. Under the specification as written, convergence is close to
  decorative unless a clade is almost exhaustively sampled. A deployment that
  wants convergence to carry real weight would need to normalise the sampling
  adequacy differently, and should say so in its pre-registration.
* Discounting a chain that rests only on reconstructed ancestral states costs a
  flat 30 % (`ρ_rec = 0.7`) and is by far the largest single effect here.
* **None of the first five scenarios changes the decision.** Even a chain whose
  mechanism has essentially no support and which rests entirely on
  reconstructions still clears `τ_E`, for the reason panel 8(b) showed: a
  weighted mean over four edges cannot be sunk by one weak edge. Only the last
  row, which re-weights the gate towards the mechanism edge, abstains.

The honest summary is that with the shipped defaults the evidence score is
driven by the raw edge supports, and the evolutionary-context terms — the part
that is novel — move it by a few percent. Whether that is the intended balance
is a question for the pre-registration, and it is exactly the question an
ablation is supposed to raise before any physical build.


In [ ]:
# (d) the search-side ablation: widen the analog prior until it is uninformative.
spreads = {
    "arm C prior (as pre-registered)": (0.20, 0.15, 0.15, 0.12),
    "prior widened x2": (0.40, 0.30, 0.30, 0.24),
    "prior widened x4 (~uniform)": (0.80, 0.60, 0.60, 0.48),
}
fig, ax = plt.subplots(figsize=(7.0, 3.4))
for (label, spread), colour in zip(spreads.items(), [RED, SAND, GREY]):
    curves = []
    for replicate in range(10):
        result = optimize.search(
            evaluator,
            lambda n, rng, spread=spread: design.analog_prior(n, spec, rng, spread=spread),
            lo, hi, population_size=40, generations=25,
            seed=stable_seed("T1", "C", replicate), arm="C",
        )
        curves.append(result.feasible_fraction_per_generation)
    curves = np.array(curves)
    ax.plot(np.arange(1, curves.shape[1] + 1), curves.mean(axis=0),
            color=colour, lw=1.6, label=label)
    ax.fill_between(np.arange(1, curves.shape[1] + 1), curves.min(axis=0), curves.max(axis=0),
                    color=colour, alpha=0.13, lw=0)
ax.set_xlabel("generation"); ax.set_ylabel("feasible fraction")
ax.legend(fontsize=7.5)
show("(d) the prior is the only channel through which the graph reaches the search")


### The phylogenetic block split

The last guard against fooling yourself. If analogs from held-out clades were
allowed into the search, retrieval could succeed by memorising close relatives
that share traits by descent. Splitting by **clade** rather than by taxon is the
phylogenetic analogue of group-wise cross-validation.


In [ ]:
train, holdout = kg.block_split(["Bambusoideae"])
print(f"full graph     : {kg.independent_origins('sandwich_stiffening')} independent origins")
print(f"training graph : {train.independent_origins('sandwich_stiffening')} "
      f"(the bamboo lineage is held out)")
print(f"held-out block : {sorted(n for n in holdout.graph if holdout.node_type(n).value == 'Organism')}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
draw_kg(train, axes[0], "training block")
draw_kg(holdout, axes[1], "held-out block (Bambusoideae)")
show()


## 11. Reproducibility and export (§5, §7.8)

Everything above is anchored to two identifiers: the **corpus snapshot hash**
(blake2b over the canonical serialisation of the corpus and its parameters) and
the **content-derived seeds**. A recommendation always refers to the snapshot
under which it was generated, so a result can be traced to the exact state of
the evidence that produced it.

The cell below runs the case study end to end and exports the per-prototype
deliverable of §6.4 — design vector, modelled predictions, traceability chain
with its four provenance records, `E`, `U` and the decision — as JSON.


In [ ]:
from evoproto.casestudy import run_case_study

case = run_case_study()
print("corpus snapshot:", case["snapshot_hash"])
print("seed           :", case["seed"])
print("abstention rate:", case["abstention_rate"], "\n")

for prototype in case["prototypes"]:
    d, p, g = prototype["design"], prototype["predictions_modeled"], prototype["gate"]
    print(f"  b = {d['b']*1e3:5.1f} mm  h = {d['h']*1e3:5.1f} mm  t = {d['t']*1e3:4.2f} mm  "
          f"rho = {d['rho']:.3f}  ->  {p['mass_kg']*1e3:6.1f} g, {p['deflection_m']*1e6:6.1f} um  "
          f"[{p['tag']}]  E = {g['E']:.3f}  U = {g['U']:.3f}  {g['decision']}")

with open("case_study.json", "w", encoding="utf-8") as handle:
    json.dump(case, handle, indent=2, default=str)
print("\nwritten: case_study.json")
print(textwrap.fill(case["note"], 88))


In [ ]:
# Re-running with the same inputs reproduces the identifiers exactly.
again = run_case_study()
print("snapshot hash stable across runs:", again["snapshot_hash"] == case["snapshot_hash"])
print("designs identical              :",
      [p["design"] for p in again["prototypes"]] == [p["design"] for p in case["prototypes"]])


In [ ]:
# The eight figures of the paper, regenerated from the same code paths.
from evoproto import figures

paths = figures.generate_all("figures")
print("\n".join(paths))


## What to take away

1. **Evolutionary context enters through priors and constraints, never through
   the objective function.** That is what makes the framework compatible with
   any certified evaluator and any established optimiser — adoption does not
   require replacing an engineering toolchain.
2. **Convergence raises confidence; it never replaces verification.** Natural
   selection optimises locally under historical contingency, so a mechanism that
   evolved three times independently is evidence of repeated *adequacy*, not of
   optimality.
3. **The abstention branch is the point, not a fallback.** A framework that
   always answers cannot be trusted with a production line; one that names the
   missing measurement can be.
4. **Nothing here has been tested against reality yet.** The dry run validates
   the pipeline; the hypotheses H1–H3 remain open until the protocol is run with
   an industrial partner, and the paper states exactly what would falsify them.

### Next steps in the repository

* `evoproto info` — every pre-registered constant, as shipped
* `evoproto dry-run --out results.json` — the run from section 9
* `evoproto case-study` — the run from section 11
* `pytest` — 119 tests, including the analytic checks used above
* `docs/paper_mapping.md` — each equation mapped to its function and its test
* `docs/preregistration.md` — the template to fill in before a physical build

### Citing

See `CITATION.cff`. Please cite both the software archive and the article.
